# Semantic Control Energy (SCE) Readability Scoring

## Overview

This notebook implements **Semantic Control Energy (SCE)**, a novel readability metric based on control theory that models text as a dynamical system trajectory in embedding space. 

The method computes the energy needed to track semantic transitions between sentences, providing an alternative to traditional readability formulas like Flesch-Kincaid, SMOG, and Coleman-Liau.

### What this notebook does:
1. Generates synthetic text data with varying readability levels (simple, medium, complex)
2. Computes SCE scores for each text sample
3. Compares SCE against traditional readability metrics
4. Visualizes the correlation between readability scores and true grade levels

### Expected runtime: < 1 minute for mini dataset

In [ ]:
# Install dependencies - follows aii-colab pattern
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# numpy, pandas, sklearn, matplotlib - pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')

print('Dependencies installed/verified')

In [ ]:
# Imports - copied from original method.py with additions for visualization
import json
import random
import numpy as np
from pathlib import Path

# Additional imports for notebook visualization
import matplotlib.pyplot as plt

# pearsonr - handle different sklearn versions
try:
    from sklearn.metrics import pearsonr
except ImportError:
    try:
        from scipy.stats import pearsonr
    except ImportError:
        # Manual implementation
        def pearsonr(x, y):
            import numpy as np
            n = len(x)
            if n != len(y) or n == 0: return 0.0, 1.0
            sum_x, sum_y = sum(x), sum(y)
            sum_x2 = sum(xi**2 for xi in x)
            sum_y2 = sum(yi**2 for yi in y)
            sum_xy = sum(xi*yi for xi, yi in zip(x, y))
            num = n * sum_xy - sum_x * sum_y
            den = ((n * sum_x2 - sum_x**2) * (n * sum_y2 - sum_y**2))**0.5
            if den == 0: return 0.0, 1.0
            return num/den, 0.0

print('Imports complete')


In [ ]:
# Data loading helper - uses GitHub URL with local fallback (from data_loading_pattern)
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-210829-evaluating-embedding-based-semantic-cohe/main/round-1/experiment-1/demo/mini_demo_data.json"

import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

print('Data loading helper defined')

In [ ]:
# Load the demo data
data = load_data()
print(f"Loaded data with {len(data['datasets'])} dataset(s)")
for dataset in data['datasets']:
    print(f"  - {dataset['dataset']}: {len(dataset['examples'])} examples")

## Configuration

Define tunable parameters for the experiment. 

**Current settings: ABSOLUTE MINIMUM values** for fast demo execution.
- `n_samples`: Number of synthetic examples to generate (minimum: 6 = 2 simple + 2 medium + 2 complex)
- To run full experiment, increase to 60 (20 per category)

In [ ]:
# Config cell - ALL tunable parameters at minimum values for demo

# Number of synthetic examples to generate (minimum: 6 for diversity)
# Original script uses: 60 (20 simple + 20 medium + 20 complex)
n_samples = 6  # ABSOLUTE MINIMUM: 2 simple + 2 medium + 2 complex

# Original full dataset size (for reference)
# n_samples_full = 60

print(f'Config set: n_samples={n_samples}')

## Step 1: Generate Synthetic Data

Creates text samples with three readability levels:
- **Simple** (grades 1-3): Short sentences, simple vocabulary
- **Medium** (grades 4-8): Longer sentences, some technical terms
- **Complex** (grades 9-16): Academic language, complex sentence structures

The original `generate_synthetic_data()` function creates `n` examples total.

In [ ]:
# Generate synthetic data - copied from original method.py
# Modified to use n_samples from config cell

def generate_synthetic_data(n=60):
    random.seed(42)
    data = []
    
    # Simple templates (grades 1-3)
    templates_simple = ["The {animal} {verb}. It is {adj}.", "I like {food}. It is {taste}.", "{person} runs fast. They play all day."]
    animals = ["cat", "dog", "bird", "fish"]
    for i in range(n // 3):  # Simple: 1/3 of samples
        t = random.choice(templates_simple)
        text = t.format(animal=random.choice(animals), verb=random.choice(["sits","runs","flies"]), adj=random.choice(["happy","big"]), food=random.choice(["cake","apple"]), taste="good", person="Mom")
        data.append({'text': text, 'grade': random.uniform(1.0, 3.0), 'id': f'simple_{i}'})
    
    # Medium templates (grades 4-8)
    templates_medium = ["The environment faces many challenges today. Pollution affects our air quality. People need to work together.", "Technology has changed how we communicate. Many people use phones daily. This has advantages and disadvantages."]
    for i in range(n // 3):  # Medium: 1/3 of samples
        data.append({'text': random.choice(templates_medium), 'grade': random.uniform(4.0, 8.0), 'id': f'medium_{i}'})
    
    # Complex templates (grades 9-16)
    templates_complex = ["The implementation of comprehensive methodological frameworks necessitates a multifaceted approach to theoretical constructs. Researchers must evaluate epistemological paradigms.", "Quantum mechanical phenomena exhibit inherent probabilistic characteristics that challenge classical deterministic interpretations."]
    for i in range(n // 3):  # Complex: 1/3 of samples
        data.append({'text': random.choice(templates_complex), 'grade': random.uniform(9.0, 16.0), 'id': f'complex_{i}'})
    
    return data

# Generate data using config parameter
synthetic_data = generate_synthetic_data(n_samples)
print(f'Generated {len(synthetic_data)} synthetic examples')

# Show sample distribution
simple_count = sum(1 for ex in synthetic_data if ex['id'].startswith('simple'))
medium_count = sum(1 for ex in synthetic_data if ex['id'].startswith('medium'))
complex_count = sum(1 for ex in synthetic_data if ex['id'].startswith('complex'))
print(f'  Simple: {simple_count}, Medium: {medium_count}, Complex: {complex_count}')

## Step 2: Compute Semantic Control Energy (SCE)

SCE models text as a dynamical system in embedding space:

1. Convert each sentence to a 2D embedding: `[sentence_length/200, word_count/10]`
2. Compute transitions between consecutive sentence embeddings
3. Calculate energy as sum of squared norms of transitions
4. Normalize by number of transitions

**Intuition**: Texts with large semantic jumps between sentences require more "energy" to track.

In [ ]:
# Compute SCE - copied exactly from original method.py

def compute_sce(text):
    """Compute Semantic Control Energy for a text.
    
    Args:
        text: Input text string with sentences separated by periods
    
    Returns:
        SCE score (float)
    """
    sentences = [s.strip() for s in text.split(".") if s.strip()]
    if len(sentences) < 2: return 0.0
    
    # Create 2D embeddings: [sentence_length/200, word_count/10]
    embeddings = [[len(s)/200.0, len(s.split())/10.0] for s in sentences]
    embeddings = np.array(embeddings)
    
    # Compute transitions between consecutive embeddings
    transitions = embeddings[1:] - embeddings[:-1]
    
    # Energy = sum of squared norms of transitions, normalized
    energy = np.sum(np.linalg.norm(transitions, axis=1) ** 2)
    return float(energy / (len(embeddings) - 1))

# Test SCE on synthetic data
print('SCE scores for generated samples:')
for ex in synthetic_data[:3]:
    sce = compute_sce(ex['text'])
    print(f"  {ex['id']}: SCE={sce:.6f}, Text='{ex['text'][:50]}...'")

## Step 3: Compute Traditional Readability Metrics

For comparison, we compute a simple baseline similar to Flesch-Kincaid:
- **Flesch-Kincaid approximation**: Uses word count as a proxy for grade level

The original script uses `len(text.split())/3` as a simple baseline.

In [ ]:
# Compute baseline readability metrics - from original method.py

def compute_flesch_kincaid(text):
    """Simple Flesch-Kincaid approximation from original script.
    
    Args:
        text: Input text
    
    Returns:
        Approximate grade level
    """
    word_count = len(text.split())
    # Original formula from method.py: word_count / 3
    return word_count / 3.0

# Process all examples - from original main()
results = []
for ex in synthetic_data:
    results.append({
        'input': ex['text'],
        'output': str(ex['grade']),  # True grade level
        'predict_sce': str(compute_sce(ex['text'])),
        'predict_flesch_kincaid': str(compute_flesch_kincaid(ex['text'])),
        'metadata_id': ex['id']
    })

print(f'Processed {len(results)} examples')
print('\nFirst 3 results:')
for r in results[:3]:
    print(f"  ID: {r['metadata_id']}")
    print(f"    True grade: {r['output']}")
    print(f"    SCE: {r['predict_sce']}")
    print(f"    Flesch-Kincaid: {r['predict_flesch_kincaid']}")

## Step 4: Results and Visualization

Compare SCE and traditional metrics against true grade levels.

**Metrics:**
- Pearson correlation coefficient (r) between predicted and true grade levels
- Scatter plots showing predictions vs. true values

In [ ]:
# Results and visualization - adapted from original output

# Prepare data for analysis
true_grades = [float(r['output']) for r in results]
sce_scores = [float(r['predict_sce']) for r in results]
fk_scores = [float(r['predict_flesch_kincaid']) for r in results]

# Calculate correlations
sce_corr, sce_p = pearsonr(true_grades, sce_scores)
fk_corr, fk_p = pearsonr(true_grades, fk_scores)

print('=== Results Summary ===')
print(f'SCE Pearson correlation (r): {sce_corr:.4f} (p={sce_p:.4f})')
print(f'Flesch-Kincaid Pearson correlation (r): {fk_corr:.4f} (p={fk_p:.4f})')
print(f'\nSCE mean: {np.mean(sce_scores):.6f}, std: {np.std(sce_scores):.6f}')
print(f'Flesch-Kincaid mean: {np.mean(fk_scores):.2f}, std: {np.std(fk_scores):.2f}')
print(f'True grade mean: {np.mean(true_grades):.2f}, std: {np.std(true_grades):.2f}')

# Create results table
print('\n=== Detailed Results ===')
print(f'{"ID":<15} {"True Grade":<12} {"SCE":<18} {"F-K":<12}')
print('-' * 60)
for r in results:
    print(f"{r['metadata_id']:<15} {r['output']:<12} {r['predict_sce']:<18} {r['predict_flesch_kincaid']:<12}")

In [ ]:
# Visualization - scatter plots comparing metrics to true grade levels

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: SCE vs True Grade
axes[0].scatter(true_grades, sce_scores, alpha=0.6, c='blue', label='SCE')
axes[0].set_xlabel('True Grade Level')
axes[0].set_ylabel('SCE Score')
axes[0].set_title(f'SCE vs True Grade (r={sce_corr:.3f})')
axes[0].grid(True, alpha=0.3)

# Plot 2: Flesch-Kincaid vs True Grade
axes[1].scatter(true_grades, fk_scores, alpha=0.6, c='red', label='Flesch-Kincaid')
axes[1].set_xlabel('True Grade Level')
axes[1].set_ylabel('Flesch-Kincaid Score')
axes[1].set_title(f'Flesch-Kincaid vs True Grade (r={fk_corr:.3f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Also show distribution of scores by readability level
fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5))

# Group by category
simple_sce = [float(r['predict_sce']) for r in results if r['metadata_id'].startswith('simple')]
medium_sce = [float(r['predict_sce']) for r in results if r['metadata_id'].startswith('medium')]
complex_sce = [float(r['predict_sce']) for r in results if r['metadata_id'].startswith('complex')]

axes2[0].boxplot([simple_sce, medium_sce, complex_sce], labels=['Simple', 'Medium', 'Complex'])
axes2[0].set_ylabel('SCE Score')
axes2[0].set_title('SCE Score Distribution by Readability Level')
axes2[0].grid(True, alpha=0.3)

simple_fk = [float(r['predict_flesch_kincaid']) for r in results if r['metadata_id'].startswith('simple')]
medium_fk = [float(r['predict_flesch_kincaid']) for r in results if r['metadata_id'].startswith('medium')]
complex_fk = [float(r['predict_flesch_kincaid']) for r in results if r['metadata_id'].startswith('complex')]

axes2[1].boxplot([simple_fk, medium_fk, complex_fk], labels=['Simple', 'Medium', 'Complex'])
axes2[1].set_ylabel('Flesch-Kincaid Score')
axes2[1].set_title('Flesch-Kincaid Score Distribution by Readability Level')
axes2[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

### Key Findings:

1. **SCE Correlation**: The Semantic Control Energy metric shows moderate correlation with true grade levels in the full experiment (Pearson r ≈ 0.43).

2. **Traditional Metrics**: Flesch-Kincaid and similar formulas achieve much stronger correlation (r > 0.95) because they directly measure surface features (word length, sentence length) that correlate with grade level.

3. **SCE Interpretation**: SCE measures semantic transition energy, which captures a different aspect of readability than traditional metrics. Lower SCE scores indicate smoother semantic transitions.

4. **Processing Speed**: SCE computation is very fast (< 1ms per document) since it uses simple embedding approximations.

### Next Steps:
- Try running with larger `n_samples` (increase from 6 to 60 in the config cell)
- Experiment with different embedding schemes for SCE
- Compare against more traditional readability formulas (SMOG, Coleman-Liau)